# Understanding Embeddings and Vector Search

## How Our RAG System Finds Relevant Documents

This notebook explains embeddings (converting text to vectors) and how FAISS uses them to find similar documents.

---

## 1. What are Embeddings?

### The Problem

Computers can't understand text like humans do. When you ask "How to create PO?", the computer sees just characters: H, o, w, space, t, o, ...

**How do we make computers understand meaning?**

### The Solution: Embeddings

Embeddings convert text to **vectors** (lists of numbers). Similar words get similar numbers.

In [ ]:
# Example: Simple embedding (in reality, it's 768 numbers!)
# We'll use 5 numbers for simplicity

embeddings = {
    "purchase order": [0.8, -0.2, 0.5, 0.1, -0.3],
    "PO creation": [0.79, -0.21, 0.51, 0.09, -0.29],  # Very similar to "purchase order"
    "invoice": [0.2, 0.9, -0.5, 0.3, 0.7],             # Different
    "goods receipt": [0.75, -0.1, 0.4, 0.15, -0.25],  # Similar to PO
}

for word, vector in embeddings.items():
    print(f"{word:20} → {vector}")

**Notice:** "purchase order" and "PO creation" have similar numbers!

That's the key: similar meanings → similar vectors

---

## 2. Measuring Similarity Between Vectors

In [ ]:
import math

# Distance formula: Euclidean distance (Pythagorean theorem in N dimensions)
def euclidean_distance(v1, v2):
    """Calculate distance between two vectors"""
    if len(v1) != len(v2):
        raise ValueError("Vectors must have same length")
    
    # Sum of squared differences
    sum_squares = sum((a - b) ** 2 for a, b in zip(v1, v2))
    
    # Take square root
    return math.sqrt(sum_squares)

# Test
v1 = [0.8, -0.2, 0.5, 0.1, -0.3]
v2 = [0.79, -0.21, 0.51, 0.09, -0.29]  # Similar to v1
v3 = [0.2, 0.9, -0.5, 0.3, 0.7]        # Different

distance_similar = euclidean_distance(v1, v2)
distance_different = euclidean_distance(v1, v3)

print(f"Distance to similar word: {distance_similar:.4f} ← Small (close)")
print(f"Distance to different word: {distance_different:.4f} ← Large (far)")

### The Concept

Think of embeddings as points on a map:
- "purchase order" at position [0.8, -0.2, 0.5, 0.1, -0.3]
- "PO creation" very close to it
- "invoice" far away

Close vectors = similar meaning ✓

---

## 3. Our RAG System in Action

In [ ]:
# Simulating our knowledge base
knowledge_base = [
    {"id": 1, "text": "P2P process includes creating purchase orders in MM transaction", "source": "P2P Guide"},
    {"id": 2, "text": "Goods receipt is confirmed when materials arrive", "source": "P2P Guide"},
    {"id": 3, "text": "Finance process uses budget allocation codes", "source": "Finance Guide"},
    {"id": 4, "text": "Creating PO requires vendor master data", "source": "SAP Config"},
    {"id": 5, "text": "Order-to-Cash includes invoicing customers", "source": "O2C Guide"},
]

# Simulate embeddings (simplified)
print("Knowledge Base:")
for doc in knowledge_base:
    print(f"  {doc['id']}. {doc['text'][:50]}... ({doc['source']})")

In [ ]:
# Now simulate what happens when user asks: "How to create a purchase order?"

user_question = "How to create a purchase order?"

# Step 1: Calculate similarity scores (like FAISS does)
similarities = {
    1: 0.95,  # Doc1: Very similar (talks about creating PO)
    2: 0.45,  # Doc2: Not related (goods receipt)
    3: 0.10,  # Doc3: Not related (finance)
    4: 0.92,  # Doc4: Very similar (creating PO)
    5: 0.15,  # Doc5: Not related (O2C)
}

print(f"User question: '{user_question}'\n")
print("Similarity scores:")
for doc_id, score in sorted(similarities.items(), key=lambda x: x[1], reverse=True):
    doc = knowledge_base[doc_id - 1]
    bar = "█" * int(score * 20)
    print(f"  Doc {doc_id}: {score:.2f} {bar} {doc['text'][:40]}...")

In [ ]:
# Step 2: Select top 3 documents
top_k = 3
selected_docs = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:top_k]

print(f"Selected top {top_k} documents:\n")
for rank, (doc_id, score) in enumerate(selected_docs, 1):
    doc = knowledge_base[doc_id - 1]
    print(f"{rank}. (Score: {score:.2f})")
    print(f"   Text: {doc['text']}")
    print(f"   Source: {doc['source']}\n")

---

## 4. What is FAISS?

### The Problem with Naive Search

If we have 1 million documents, calculating distance to each one:
- 1 million distance calculations
- For each: compute 768 numbers
- Total: 768 million operations! 🐌

### FAISS Solution

FAISS (Facebook AI Similarity Search) uses intelligent data structures to find similar vectors **fast**.

Like an index in a book: instead of reading every page to find "purchase orders", just look in the index.

In [ ]:
# Simulating FAISS behavior

class SimpleFAISS:
    """Simplified version of FAISS for understanding"""
    
    def __init__(self, vectors):
        """Initialize with training data"""
        self.vectors = vectors  # Store all document vectors
        print(f"Index created with {len(vectors)} documents")
    
    def search(self, query_vector, k=3):
        """Find k most similar vectors"""
        distances = []
        
        for idx, vec in enumerate(self.vectors):
            # Calculate distance
            dist = euclidean_distance(query_vector, vec)
            distances.append((idx, dist))
        
        # Sort by distance (closest first)
        distances.sort(key=lambda x: x[1])
        
        # Return top k
        return distances[:k]

# Test
vectors = [
    [0.8, -0.2, 0.5, 0.1, -0.3],    # 0: PO-related
    [0.2, 0.9, -0.5, 0.3, 0.7],     # 1: Finance-related
    [0.79, -0.19, 0.51, 0.09, -0.29], # 2: PO-related
    [0.15, 0.85, -0.45, 0.25, 0.75],  # 3: Finance-related
]

index = SimpleFAISS(vectors)

# Search for documents similar to "purchase order" vector
query = [0.8, -0.2, 0.5, 0.1, -0.3]
results = index.search(query, k=2)

print("\nSearch results (k=2):")
for rank, (doc_id, distance) in enumerate(results, 1):
    print(f"  {rank}. Document {doc_id}: distance={distance:.4f}")

---

## 5. How Real Embeddings Work (In Our Project)

### From backend/app/retrieval.py

In [ ]:
# Our actual code (simplified)

class RAGRetriever:
    def __init__(self, documents):
        self.documents = documents
        # In real code: self.index = faiss.IndexFlatL2(768)
        self.index = SimpleFAISS([
            [i * 0.1 + j * 0.01 for j in range(5)] 
            for i in range(len(documents))
        ])
    
    def retrieve(self, question, k=3):
        """
        This is what happens when user asks a question:
        1. Embed the question
        2. Search for similar documents
        3. Return them as context
        """
        # Simulate embedding (in real: use Google Gemini API)
        question_embedding = [0.8, -0.2, 0.5, 0.1, -0.3]
        
        # Search
        results = self.index.search(question_embedding, k=k)
        
        # Format results
        context = []
        for rank, (doc_id, _distance) in enumerate(results):
            context.append(self.documents[doc_id])
        
        return context

# Test
docs = [
    "P2P process includes creating purchase orders in MM transaction",
    "Goods receipt is confirmed when materials arrive",
    "Creating PO requires vendor master data",
]

retriever = RAGRetriever(docs)
context = retriever.retrieve("How to create PO?", k=2)

print("Retrieved context for: 'How to create PO?'\n")
for i, doc in enumerate(context, 1):
    print(f"{i}. {doc}")

---

## 6. Real Numbers: Embedding Dimensions

In [ ]:
# In our project:
# - We use Google Gemini's embedding model: text-embedding-004
# - It creates 768-dimensional vectors
# - That means 768 numbers per document!

print("Real embedding sizes in different models:")
print()
print("Model              | Dimensions | Speed       | Accuracy")
print("-" * 60)
print("text-embedding-004 | 768        | Very Fast   | Good")
print("BERT               | 768        | Medium      | Excellent")
print("GPT-4 Embeddings   | 1536       | Slow        | Excellent")
print()
print("We use text-embedding-004 because:")
print("  ✓ Fast - Good for real-time responses")
print("  ✓ Free tier - From Google")
print("  ✓ Good accuracy - Fine for ERP domain")

---

## Summary: How Our RAG System Uses Embeddings

1. **User asks**: "How to create PO?"
2. **Convert to vector**: 768 numbers representing meaning
3. **Search with FAISS**: Find 3 closest document vectors
4. **Add context**: Found docs go into prompt
5. **AI responds**: Gemini reads prompt with context and generates answer

**Result**: AI gives accurate, sourced answers! 🎯